# Individual Programming Assignment - Song Recommender System
CMPE 256, Spring 2025

Kaushika Uppu, 014756859


## Introduction

With personalized content becoming more and more universal as artificial intelligence and machine learning continue to expand into various industries and fields, song recommender systems can truly enhance user experience by suggesting more relevant music. The focus of this programming project was to implement a song recommender system that can predict a user's rating for a song given a dataset containing user ids, song ids, and song metadata. This project uses a Neural Collaborative Filtering (NCF) approach that combines user, song, and semantic metadata embeddings to learn user-song interactions.

## Imports

In [104]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

First, all of the data was imported.

### Training Dataset

In [16]:
training_data = pd.read_csv("train.csv")
training_data.head()

,user_id,song_id,rating
0,1257279,1133436,4.25
1,1521617,1041044,2.75
2,1757741,1018376,4.25
3,1311545,1035650,5.25
4,1633733,1080634,8.25


### Testing Dataset

In [17]:
testing_data = pd.read_csv("test.csv")
testing_data.head()

,user_id,song_id
0,1717534,1005189
1,1302257,1042789
2,1700269,1042495
3,1265736,1040200
4,1060963,1008334


### Song Metadata

In [18]:
song_data = pd.read_csv("song_data.csv")
song_data = song_data.drop_duplicates(subset=['song_id'])
song_data.head()

,song_id,title,release,artist_name,year
0,1191338,Silent Night,Monster Ballads X-Mas,Faster Pussy cat,2003
1,1178322,L'antarctique,Des cobras des tarentules,3 Gars Su'l Sofa,2007
2,1052052,Ethos of Coercion,Descend Into Depravity,Dying Fetus,2009
3,1072961,Rock-N-Rule,I'm Only A Man (Bonus Track Version),Emery,2007
4,1086060,All of the same blood,Violent revolution,Kreator,2001


## Data Preprocessing

### Encoding User and Song IDs

First, all unique user and song IDs seen in both training and testing datasets were encoded using two separate `LabelEncoder` instances.

In [19]:
user_encoder = LabelEncoder()
song_encoder = LabelEncoder()

In [20]:
all_user_ids = list(set(list(training_data['user_id']) + list(testing_data['user_id'])))
user_encoder.fit(all_user_ids)

LabelEncoder()

In [21]:
all_song_ids = list(set(song_data['song_id']))
song_encoder.fit(all_song_ids)

LabelEncoder()

In [22]:
# encoding user and song ids
training_data['user_id_encoded'] = user_encoder.transform(training_data['user_id'])
training_data['song_id_encoded'] = song_encoder.transform(training_data['song_id'])
testing_data['user_id_encoded'] = user_encoder.transform(testing_data['user_id'])
testing_data['song_id_encoded'] = song_encoder.transform(testing_data['song_id'])

The feature `user_id-song_id` was created in the testing dataset, as it will be the index needed for the submission file.

In [23]:
testing_data['user_id-song_id'] = [str(testing_data['user_id'].values[i]) + "-" + str(testing_data['song_id'].values[i]) 
                                   for i in range(len(testing_data))]

### Metadata

Next, the metadata needs to be transformed into a format that can be inputted into the model.

#### Training Data

In [24]:
training_data = training_data.merge(song_data[['song_id', 'title', 'year', 'release', 'artist_name']],
                                    on = 'song_id', how = 'left')
training_data = training_data.drop_duplicates()

The features `title`, `release`, `artist_name`, and `year` all need to be feature engineered. For the first three, `SentenceTransformer` was used; this resulted in embeddings that captured the semantic meaning of the features, which could help the model learn similarities and relationships between songs.

Each of the three features were embedded separately. First, the pre-trained `all-MiniLM-L6-v2` model was loaded in. Then, all unique values for the feature were encoded into embeddings using the transformer.

In [25]:
# training title embeddings
title_transformer = SentenceTransformer('all-MiniLM-L6-v2')
unique_titles = training_data['title'].fillna('Unknown').unique()
unique_title_embeddings = title_transformer.encode(unique_titles.tolist(), batch_size = 256, 
                                                   convert_to_numpy = True, show_progress_bar = True)

Batches:   0%|          | 0/596 [00:00<?, ?it/s]

Then, since only unique titles were embedded above, a dictionary that mapped each title to its embedding was created. This was then used to create a list of title embeddings for the entire training set.

In [26]:
title_to_emb = dict(zip(unique_titles, unique_title_embeddings))
title_embeddings = np.vstack(training_data['title'].fillna('Unknown').map(title_to_emb).to_list())

Finally, as seen below, `SentenceTransformer` creates dense embeddings of length 384. Therefore, they were then reduced using `PCA` with 5 components for the `title` feature. These embeddings were then concatenated back with the training dataset.

In [152]:
len(title_embeddings[0])

384

In [27]:
# reducing training title embeddings into 5 principal components
title_pca = PCA(n_components = 5)
reduced_title_features_train = title_pca.fit_transform(title_embeddings)
title_features_df = pd.DataFrame(reduced_title_features_train, 
                                   columns=[f'title_PC{i+1}' for i in range(reduced_title_features_train.shape[1])])

training_data = pd.concat([training_data, title_features_df], axis=1)

This process was then repeated for the `release` (3 components for PCA) and `artist_name` (3 components) features for the training dataset.

In [28]:
# training release embeddings
release_transformer = SentenceTransformer('all-MiniLM-L6-v2')
unique_releases = training_data['release'].fillna('Unknown').unique()
unique_release_embeddings = release_transformer.encode(unique_releases.tolist(), batch_size = 256, 
                                                       convert_to_numpy = True, show_progress_bar = True)

Batches:   0%|          | 0/188 [00:00<?, ?it/s]

In [29]:
release_to_emb = dict(zip(unique_releases, unique_release_embeddings))
release_embeddings = np.vstack(training_data['release'].fillna('Unknown').map(release_to_emb).to_list())

In [30]:
# reducing training release embeddings into 3 principal components
release_pca = PCA(n_components = 3)
reduced_release_features_train = release_pca.fit_transform(release_embeddings)

release_features_df = pd.DataFrame(reduced_release_features_train, 
                                   columns=[f'release_PC{i+1}' for i in range(reduced_release_features_train.shape[1])])

training_data = pd.concat([training_data, release_features_df], axis=1)

In [31]:
# training artist name embeddings
artist_transformer = SentenceTransformer('all-MiniLM-L6-v2')
unique_artists = training_data['artist_name'].fillna('Unknown').unique()
unique_artist_embeddings = artist_transformer.encode(unique_artists.tolist(), batch_size = 256, 
                                                   convert_to_numpy = True, show_progress_bar = True)

Batches:   0%|          | 0/95 [00:00<?, ?it/s]

In [32]:
artist_to_emb = dict(zip(unique_artists, unique_artist_embeddings))
artist_embeddings = np.vstack(training_data['artist_name'].fillna('Unknown').map(artist_to_emb).to_list())

In [33]:
# reducing training artist name embeddings into 3 principal components
artist_pca = PCA(n_components = 3)
reduced_artist_features_train = artist_pca.fit_transform(artist_embeddings)

artist_features_df = pd.DataFrame(reduced_artist_features_train, 
                                   columns=[f'artist_PC{i+1}' for i in range(reduced_artist_features_train.shape[1])])

training_data = pd.concat([training_data, artist_features_df], axis=1)

Moving onto the `year` variable, the first feature that was created a normalized year feature, `year_norm`. This used a `MinMaxScaler` to normalize years to a 0-1 scale. 

In [34]:
# normalizing year to 0-1 scale
year_scaler = MinMaxScaler()
training_data['year_norm'] = year_scaler.fit_transform(training_data[['year']])

Then, a `decade` feature was created. This was then used to create `decade_id` with a `LabelEncoder`, which treats decade as a categorical ID feature from 0-9, which represents the decades 1920s-2010s.

In [35]:
# creating categorical decade ids
training_data['decade'] = (training_data['year'] // 10) * 10
decade_encoder = LabelEncoder()
training_data['decade_id'] = decade_encoder.fit_transform(training_data['decade'])

In [36]:
training_data = training_data.drop(columns = ['title', 'year', 'release', 'artist_name', 'decade'])

In [37]:
training_data.head()

,user_id,song_id,rating,user_id_encoded,song_id_encoded,title_PC1,title_PC2,title_PC3,title_PC4,title_PC5,release_PC1,release_PC2,release_PC3,artist_PC1,artist_PC2,artist_PC3,year_norm,decade_id
0,1257279,1133436,4.25,253216,133436,-0.104249,-0.133150,0.055198,0.130816,0.220293,-0.033660,-0.129120,-0.089329,-0.109292,-0.003296,0.008325,0.887640,8
1,1521617,1041044,2.75,513406,41044,-0.180311,0.037817,0.135498,-0.119857,0.005868,-0.201560,-0.078446,0.127966,0.125683,-0.067910,-0.125200,0.955056,8
2,1757741,1018376,4.25,745828,18376,0.000277,0.023894,-0.088176,-0.079743,0.085656,-0.145385,-0.082999,0.078127,0.013056,0.182955,0.163267,0.887640,8
3,1311545,1035650,5.25,306619,35650,-0.129985,-0.030214,-0.013668,-0.003569,0.141529,0.243068,0.060677,0.033351,0.270285,-0.008725,0.282341,0.921348,8
4,1633733,1080634,8.25,623814,80634,-0.104444,0.422260,-0.295674,-0.058879,0.087214,0.198222,-0.053847,0.023648,-0.068496,0.138477,-0.105286,0.943820,8


#### Testing Data

All of the metadata preprocessing was then repeated for the testing dataset so it contains all of the same features.

In [38]:
testing_data = testing_data.merge(song_data[['song_id', 'title', 'year', 'release', 'artist_name']],
                                  on = 'song_id', how = 'left')
testing_data = testing_data.drop_duplicates()

Since the `SentenceTransformers` were already created above, they did not need to be generated again, and were simply just used to get the testing dataset embeddings for the `title`, `release`, and `artist_name` variables. Furthermore, the `PCA` instances were not fit again. Instead, the same instances that were fit on the training set were used for the testing set to transform the embeddings into reduced principal components.

In [39]:
# testing title embeddings
unique_test_titles = testing_data['title'].fillna('Unknown').unique()
unique_test_title_embeddings = title_transformer.encode(unique_test_titles, batch_size = 256,
                                                        convert_to_numpy = True, show_progress_bar = True)

Batches:   0%|          | 0/375 [00:00<?, ?it/s]

In [40]:
title_to_emb_test = dict(zip(unique_test_titles, unique_test_title_embeddings))
test_title_embeddings = np.vstack(testing_data['title'].fillna('Unknown').map(title_to_emb_test).to_list())

In [41]:
# reducing testing title embeddings into 5 principal components
reduced_title_features_test = title_pca.transform(test_title_embeddings)
test_title_features_df = pd.DataFrame(reduced_title_features_test, 
                                   columns=[f'title_PC{i+1}' for i in range(reduced_title_features_test.shape[1])])

testing_data = pd.concat([testing_data, test_title_features_df], axis=1)

In [42]:
# testing release embeddings
unique_test_releases = testing_data['release'].fillna('Unknown').unique()
unique_test_release_embeddings = release_transformer.encode(unique_test_releases, batch_size = 256,
                                                        convert_to_numpy = True, show_progress_bar = True)

Batches:   0%|          | 0/148 [00:00<?, ?it/s]

In [43]:
release_to_emb_test = dict(zip(unique_test_releases, unique_test_release_embeddings))
test_release_embeddings = np.vstack(testing_data['release'].fillna('Unknown').map(release_to_emb_test).to_list())

In [44]:
# reducing testing release embeddings into 3 principal components
reduced_release_features_test = release_pca.transform(test_release_embeddings)
test_release_features_df = pd.DataFrame(reduced_release_features_test, 
                                   columns=[f'release_PC{i+1}' for i in range(reduced_release_features_test.shape[1])])

testing_data = pd.concat([testing_data, test_release_features_df], axis=1)

In [45]:
# testing artist name embeddings
unique_test_artists = testing_data['artist_name'].fillna('Unknown').unique()
unique_test_artist_embeddings = artist_transformer.encode(unique_test_artists, batch_size = 256,
                                                        convert_to_numpy = True, show_progress_bar = True)

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

In [46]:
artist_to_emb_test = dict(zip(unique_test_artists, unique_test_artist_embeddings))
test_artist_embeddings = np.vstack(testing_data['artist_name'].fillna('Unknown').map(artist_to_emb_test).to_list())

In [47]:
# reducing testing artist name embeddings into 3 principal components
reduced_artist_features_test = artist_pca.transform(test_artist_embeddings)
test_artist_features_df = pd.DataFrame(reduced_artist_features_test, 
                                   columns=[f'artist_PC{i+1}' for i in range(reduced_artist_features_test.shape[1])])

testing_data = pd.concat([testing_data, test_artist_features_df], axis=1)

The same `MinMaxScaler` and `LabelEncoder` from the training dataset were used to transform the `year` variable to `year_norm` and `decade_id` for the testing dataset.

In [48]:
# normalizing year to 0-1 scale
testing_data['year_norm'] = year_scaler.transform(testing_data[['year']])

In [49]:
# creating categorical decade ids
testing_data['decade'] = (testing_data['year'] // 10) * 10
testing_data['decade_id'] = decade_encoder.transform(testing_data['decade'])

In [50]:
testing_data = testing_data.drop(columns = ['title', 'year', 'release', 'artist_name', 'decade'])

In [53]:
testing_data.head()

,user_id,song_id,user_id_encoded,song_id_encoded,user_id-song_id,title_PC1,title_PC2,title_PC3,title_PC4,title_PC5,release_PC1,release_PC2,release_PC3,artist_PC1,artist_PC2,artist_PC3,year_norm,decade_id
0,1717534,1005189,706261,5189,1717534-1005189,-0.070450,-0.100112,0.029512,0.146849,-0.019690,-0.178449,-0.072485,-0.121277,0.007929,-0.260165,0.014176,0.977528,8
1,1302257,1042789,297483,42789,1302257-1042789,-0.022869,-0.293364,-0.125680,0.062023,0.006545,0.201598,0.111797,-0.159742,0.213903,0.048029,0.118322,0.449438,4
2,1700269,1042495,689283,42495,1700269-1042495,-0.034142,-0.039668,0.059762,0.084298,0.165493,-0.042889,-0.059624,-0.054593,-0.134922,-0.354998,-0.027317,0.910112,8
3,1265736,1040200,261542,40200,1265736-1040200,0.008757,0.098394,0.158625,0.095618,0.125722,-0.150966,-0.012326,-0.028047,-0.033501,0.070110,-0.175344,0.988764,9
4,1060963,1008334,60000,8334,1060963-1008334,-0.111860,-0.082844,-0.053006,-0.087540,-0.056769,0.277288,-0.032862,0.162603,-0.090158,0.135821,0.027052,0.955056,8


## Neural Collaborative Filtering Model

### Training

Now that all data preprocessing and feature engineering are completed, the NCF model was then built and trained. 

In [54]:
X = training_data.drop(columns=['user_id', 'song_id', 'rating'])
y = training_data['rating']

The model was built using inputs for user, song, metadata, and decade. The decade input was concatenated to the metadata input after being embedded, however. The embeddings were then passed through Dense layers so the model could understand non-linear relationships with the relu activation function. Finally, the towers were concatenated and passed through two last Dense layers. Then, the output layer uses the sigmoid activation function in order to output values between 0 and 1. The reason why this was chosen is explained below.

In [66]:
def build_model(num_users, num_songs, num_decades, metadata_dim):
    # inputs
    user_input = Input(shape = (), name = 'user_input')
    song_input = Input(shape = (), name = 'song_input')
    metadata_input = Input(shape = (metadata_dim,), name = 'metadata_input')
    decade_input = Input(shape=(), name = 'decade_input')

    # embeddings
    user_embedding = Embedding(input_dim = num_users, output_dim = 32)(user_input)
    user_vec = Flatten()(user_embedding)
    user_vec = Dense(64, activation = 'relu')(user_vec)

    song_embedding = Embedding(input_dim = num_songs, output_dim = 32)(song_input)
    song_vec = Flatten()(song_embedding)
    song_vec = Dense(64, activation = 'relu')(song_vec)

    decade_emb = Embedding(input_dim = num_decades, output_dim = 4)(decade_input)
    decade_vec = Flatten()(decade_emb)

    meta_vec = Dense(64, activation = 'relu')(metadata_input)
    meta_vec = Dense(32, activation = 'relu')(meta_vec)
    meta_combined = Concatenate()([meta_vec, decade_vec])

    # final concatenation
    combined = Concatenate()([user_vec, song_vec, meta_combined])
    x = Dense(64, activation = 'relu')(combined)
    x = Dense(32, activation = 'relu')(x)
    output = Dense(1, activation = 'sigmoid')(x)

    model = Model(inputs = [user_input, song_input, metadata_input, decade_input], outputs = output)
    return model

The output layer uses a sigmoid activation function to limit a 0-1 output because all of the ratings were scaled using a `MinMaxScaler` to a 0-1 scale, as shown in the code below. This is because this resulted in better ratings predictions with a lower RMSE compared to keeping ratings in a 1-10 scale.

In [84]:
# train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.3, random_state = 42)

# input extraction
metadata_features = ['title_PC1', 'title_PC2', 'title_PC3', 'title_PC4', 'title_PC5', 'release_PC1', 'release_PC2', 
                     'release_PC3', 'artist_PC1', 'artist_PC2', 'artist_PC3', 'year_norm']
rating_scaler = MinMaxScaler()

train_user_ids = X_train['user_id_encoded'].values
train_song_ids = X_train['song_id_encoded'].values
train_metadata = X_train[metadata_features].values
train_decades = X_train['decade_id'].values
y_train_scaled = rating_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).flatten()

val_user_ids = X_val['user_id_encoded'].values
val_song_ids = X_val['song_id_encoded'].values
val_metadata = X_val[metadata_features].values
val_decades = X_val['decade_id'].values
y_val_scaled = rating_scaler.transform(y_val.to_numpy().reshape(-1, 1)).flatten()

In [141]:
# input dimensions
num_users = training_data['user_id_encoded'].max() + 1
num_songs = training_data['song_id_encoded'].max() + 1
num_decades = training_data['decade_id'].max() + 1
metadata_features_dim = train_metadata.shape[1]

In [86]:
model = build_model(num_users = num_users + 1,
                        num_songs = num_songs + 1,
                        num_decades = num_decades + 1,
                        metadata_dim = metadata_features_dim)

The model was then compiled and fit, using the `adam` optimizer and `mse` loss, with RMSE as the metric. `EarlyStopping` was also used during fitting to try and reduce model overfitting.

In [87]:
model.compile(optimizer='adam', loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError()])

In [88]:
model.fit(
    x = [train_user_ids, train_song_ids, train_metadata, train_decades],
    y = y_train_scaled,
    validation_data = ([val_user_ids, val_song_ids, val_metadata, val_decades], y_val_scaled),
    batch_size = 512,
    epochs = 5,
    callbacks = [tf.keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)]
)

Epoch 1/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 471s 65ms/step - loss: 0.0364 - root_mean_squared_error: 0.1908 - val_loss: 0.0358 - val_root_mean_squared_error: 0.1893
Epoch 2/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 471s 65ms/step - loss: 0.0340 - root_mean_squared_error: 0.1844 - val_loss: 0.0364 - val_root_mean_squared_error: 0.1909
Epoch 3/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 470s 65ms/step - loss: 0.0297 - root_mean_squared_error: 0.1722 - val_loss: 0.0388 - val_root_mean_squared_error: 0.1969
Epoch 4/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 467s 64ms/step - loss: 0.0242 - root_mean_squared_error: 0.1557 - val_loss: 0.0421 - val_root_mean_squared_error: 0.2051


With training complete, the model was then used to predict ratings for the validation dataset. As seen below, the ratings were first inversely transformed back to a 1-10 rating scale.

In [94]:
# scaling ratings back to 1-10 scale
val_preds_scaled = model.predict([val_user_ids, val_song_ids, val_metadata, val_decades])
val_preds = rating_scaler.inverse_transform(mt_val_preds_scaled.reshape(-1, 1)).flatten()

# evaluating on validation dataset
rmse = tf.keras.metrics.RootMeanSquaredError()
rmse.update_state(y_val, val_preds)
print("Validation RMSE:", rmse.result().numpy())

49859/49859 ━━━━━━━━━━━━━━━━━━━━ 17s 331us/step
Validation RMSE: 1.7036666


As seen above, the RMSE for the validation set was 1.703.

### Hyperparameter Tuning

Before predicting on the testing dataset, hyperparameter tuning was carried out first. The four parameters that were tuned were embedding dimension, the number of units for dense layers 1 and 2, and the learning rate for the optimizer.

In [102]:
def tune_model(hp):
    # parameters to tune
    embedding_dim = hp.Choice('embedding_dim', [16, 32, 64])
    dense_units_1 = hp.Choice('dense_1', [32, 64, 128])
    dense_units_2 = hp.Choice('dense_2', [16, 32, 64])
    learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])

    # inputs
    user_input = Input(shape = (), name = 'user_input')
    song_input = Input(shape = (), name = 'song_input')
    metadata_input = Input(shape = (train_metadata.shape[1],), name = 'metadata_input')
    decade_input = Input(shape = (), name = 'decade_input')

    # embeddings
    user_embedding = Embedding(input_dim = num_users, output_dim = embedding_dim)(user_input)
    user_vec = Flatten()(user_embedding)
    user_vec = Dense(dense_units_1, activation = 'relu')(user_vec)

    song_embedding = Embedding(input_dim = num_songs, output_dim = embedding_dim)(song_input)
    song_vec = Flatten()(song_embedding)
    song_vec = Dense(dense_units_1, activation = 'relu')(song_vec)

    decade_emb = Embedding(input_dim = num_decades, output_dim = 4)(decade_input)
    decade_vec = Flatten()(decade_emb)

    meta_vec = Dense(dense_units_1, activation = 'relu')(metadata_input)
    meta_vec = Dense(dense_units_2, activation = 'relu')(meta_vec)
    meta_combined = Concatenate()([meta_vec, decade_vec])

    # final concatenation
    combined = Concatenate()([user_vec, song_vec, meta_combined])
    x = Dense(dense_units_1, activation = 'relu')(combined)
    x = Dense(dense_units_2, activation = 'relu')(x)
    output = Dense(1, activation = 'sigmoid')(x)

    model = Model(inputs = [user_input, song_input, metadata_input, decade_input], outputs = output)
    model.compile(
        optimizer = Adam(learning_rate = learning_rate),
        loss = 'mse',
        metrics = [tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

Tuning was then conducted using KerasTuner's `RandomSearch`.

In [103]:
# hyperparameter tuning
tuner = kt.RandomSearch(
    tune_model,
    objective = 'val_root_mean_squared_error',
    max_trials = 5,
    executions_per_trial = 1,
    directory = 'tuner_logs',
    project_name = 'ncf_multitower'
)

tuner.search(
    x = [train_user_ids, train_song_ids, train_metadata, train_decades],
    y = y_train_scaled,
    validation_data = ([val_user_ids, val_song_ids, val_metadata, val_decades], y_val_scaled),
    epochs = 5,
    batch_size = 512,
    callbacks = [tf.keras.callbacks.EarlyStopping(patience = 2, restore_best_weights = True)]
)

Trial 5 Complete [00h 22m 58s]
val_root_mean_squared_error: 0.18930140137672424

Best val_root_mean_squared_error So Far: 0.18930140137672424
Total elapsed time: 03h 03m 06s


A summary of the tuning results:

In [148]:
tuner.results_summary()

Results summary
Results in tuner_logs/ncf_multitower
Showing 10 best trials
Objective(name="val_root_mean_squared_error", direction="min")

Trial 4 summary
Hyperparameters:
embedding_dim: 32
dense_1: 128
dense_2: 32
learning_rate: 0.001
Score: 0.18930140137672424

Trial 3 summary
Hyperparameters:
embedding_dim: 64
dense_1: 64
dense_2: 64
learning_rate: 0.001
Score: 0.189354807138443

Trial 0 summary
Hyperparameters:
embedding_dim: 64
dense_1: 32
dense_2: 64
learning_rate: 0.0001
Score: 0.18962214887142181

Trial 1 summary
Hyperparameters:
embedding_dim: 16
dense_1: 32
dense_2: 32
learning_rate: 0.01
Score: 0.18965968489646912

Trial 2 summary
Hyperparameters:
embedding_dim: 64
dense_1: 64
dense_2: 16
learning_rate: 0.01
Score: 0.1906503289937973


The best hyperparameters that were found are printed below:

In [128]:
best_hps = tuner.get_best_hyperparameters(1)[0]
best_hps.values

{'embedding_dim': 32, 'dense_1': 128, 'dense_2': 32, 'learning_rate': 0.001}

### Best Model

Then, the hyperparameters that were found above were used to create `best_model`, which was then trained on the training data and validated again, using the same inputs that were used before tuning.

In [129]:
# best hyperparameters that were found during tuning above
BEST_EMBEDDING_DIM = 32
BEST_DENSE_1 = 128
BEST_DENSE_2 = 32
BEST_LEARNING_RATE = 0.001

In [139]:
# inputs
user_input = Input(shape = (), name = 'user_input')
song_input = Input(shape = (), name = 'song_input')
metadata_input = Input(shape = (metadata_features_dim,), name = 'metadata_input')
decade_input = Input(shape=(), name = 'decade_input')

# embeddings
user_embedding = Embedding(input_dim = num_users + 1, output_dim = BEST_EMBEDDING_DIM)(user_input)
user_vec = Flatten()(user_embedding)
user_vec = Dense(BEST_DENSE_1, activation = 'relu')(user_vec)

song_embedding = Embedding(input_dim = num_songs + 1, output_dim = BEST_EMBEDDING_DIM)(song_input)
song_vec = Flatten()(song_embedding)
song_vec = Dense(BEST_DENSE_1, activation = 'relu')(song_vec)

decade_emb = Embedding(input_dim = num_decades + 1, output_dim = 4)(decade_input)
decade_vec = Flatten()(decade_emb)

meta_vec = Dense(BEST_DENSE_1, activation = 'relu')(metadata_input)
meta_vec = Dense(BEST_DENSE_2, activation = 'relu')(meta_vec)
meta_combined = Concatenate()([meta_vec, decade_vec])

# final concatenation
combined = Concatenate()([user_vec, song_vec, meta_combined])
x = Dense(BEST_DENSE_1, activation = 'relu')(combined)
x = Dense(BEST_DENSE_2, activation = 'relu')(x)
output = Dense(1, activation = 'sigmoid')(x)

best_model = Model(inputs = [user_input, song_input, metadata_input, decade_input], outputs = output)
best_model.compile(
        optimizer = Adam(learning_rate = BEST_LEARNING_RATE),
        loss = 'mse',
        metrics = [tf.keras.metrics.RootMeanSquaredError()]
)

In [140]:
best_model.fit(
    x = [train_user_ids, train_song_ids, train_metadata, train_decades],
    y = y_train_scaled,
    validation_data = ([val_user_ids, val_song_ids, val_metadata, val_decades], y_val_scaled),
    batch_size = 512,
    epochs = 5,
    callbacks = [tf.keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)]
)

Epoch 1/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 467s 64ms/step - loss: 0.0364 - root_mean_squared_error: 0.1907 - val_loss: 0.0358 - val_root_mean_squared_error: 0.1893
Epoch 2/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 463s 64ms/step - loss: 0.0340 - root_mean_squared_error: 0.1844 - val_loss: 0.0362 - val_root_mean_squared_error: 0.1902
Epoch 3/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 456s 63ms/step - loss: 0.0297 - root_mean_squared_error: 0.1725 - val_loss: 0.0382 - val_root_mean_squared_error: 0.1955
Epoch 4/5
7272/7272 ━━━━━━━━━━━━━━━━━━━━ 465s 64ms/step - loss: 0.0248 - root_mean_squared_error: 0.1574 - val_loss: 0.0426 - val_root_mean_squared_error: 0.2064


In [142]:
best_val_preds_scaled = mt_model.predict([val_user_ids, val_song_ids, val_metadata, val_decades])
best_val_preds = rating_scaler.inverse_transform(best_val_preds_scaled.reshape(-1, 1)).flatten()
best_rmse = tf.keras.metrics.RootMeanSquaredError()
best_rmse.update_state(y_val, best_val_preds)
print("Validation RMSE:", best_rmse.result().numpy())

49859/49859 ━━━━━━━━━━━━━━━━━━━━ 16s 320us/step
Validation RMSE: 1.7036666


As seen with the validation RMSE above, hyperparameter tuning did not really improve model performance, as it still is 1.703.

## Results

Now that the training, hyperparameter tuning, and validation has been completed, the model can now be used to predict ratings for the testing dataset.

In [143]:
# getting test set inputs
test_user_ids = testing_data['user_id_encoded'].values
test_song_ids = testing_data['song_id_encoded'].values
test_metadata = testing_data[metadata_features].values.astype(np.float32)
test_decades = testing_data['decade_id'].values

# predicting ratings and transforming back to 1-10 scale
test_preds_scaled = best_model.predict([test_user_ids, test_song_ids, test_metadata, test_decades])
test_preds = rating_scaler.inverse_transform(test_preds_scaled.reshape(-1, 1)).flatten()

33239/33239 ━━━━━━━━━━━━━━━━━━━━ 11s 341us/step


The results were then put into a dataframe, using the `user_id-song_id` variable that was created previously as the index. A glimpse at the predicted results is shown below:

In [144]:
test_results = pd.DataFrame({
    'user_id-song_id': testing_data['user_id-song_id'], 
    'rating': test_preds.flatten()
})
test_results = test_results.set_index('user_id-song_id')
test_results.head()

,rating
user_id-song_id,
1717534-1005189,5.228999
1302257-1042789,5.977572
1700269-1042495,4.900133
1265736-1040200,5.081084
1060963-1008334,5.335390


In [149]:
test_results.to_csv('solution.csv')

### Final RMSE

After submitting on Kaggle, the final RMSE of this project was **1.7043.**

## Conclusion

By combining user, song, and metadata embeddings, this project developed a song recommender system by using a Neural Collaborative Filtering approach. Although various other methods were tried throughout the development of this project, including a basic two-tower NCF model and XGBoost Regressor, the NCF model architecture with user, song, and metadata towers resulted in the overall best performance in terms of RMSE.

One major problem that recommender systems have to handle is the cold start problem, where the testing dataset contains new users and songs that were not seen in the training dataset, as was the case for this data. The hybrid approach that was used in this project, however, helped attenuate the issue slightly. More specfically, including song metadata into the model helped the cold start problem for new songs, as even if the songs were not seen in the training dataset, there was still content-based information about them that the model could potentially use to help better its predicted ratings for the testing data.

Overall, this project found that neural collaborative filtering seems to be valuable in building personalized and generalizable recommender systems. Future work could include adding in additional metadata for users, which could help with the cold start problem for users specifically. Another future suggestion would be to explore more complex deep learning models. Finally, more song metadata might lead to better performance, especially metadata about the genres of songs.

## Sources

Code skeletons that were available at the following links were used in this code for SentenceTransformer, EarlyStopping, and RandomSearch. A modified version of the code provided in the NCF demo that we did in class was also used in developing the NCF architecture in this project.

- https://sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html 

- https://sbert.net/docs/sentence_transformer/pretrained_models.html

- https://keras.io/api/callbacks/early_stopping/

- https://keras.io/keras_tuner/getting_started/#start-the-search